# 数值、日期与时间

学习目标：根据精度、随机数用途和时区要求，选择标准库工具完成可靠的数值与时间计算。

前置知识：数值类型与运算、字符串、列表、模块导入、异常处理和 with。

运行环境：Python 3.12。

环境准备：[环境配置与运行](README.md)。

工作目录：本 Notebook 所在目录；重启内核后从上到下运行。

时区示例需要 IANA 时区数据，本环境使用 tzdata 2026.3。

## 1 浮点误差与近似比较

### 1.1 显示的小数不一定是存储的精确值

float 是内置数值类型；本章侧重它的精度边界和标准库补充工具，不再重复基本运算语法。

float 使用二进制浮点表示，0.1 等十进制小数通常只能近似存储。运算可能进一步舍入，因此不能把近似计算后的相等判断当作十进制恒等式。格式化只改变显示，不修复存储值。

math.isclose 用容差判断两个数是否足够接近。rel_tol 是相对容差，按两个数中较大的绝对值缩放；默认值为 1e-9。容差应符合输入精度和任务要求。

In [1]:
import math

combined = 0.1 + 0.2
print(combined)  # 0.30000000000000004：二进制近似参与了运算。
print(f"{combined:.1f}")  # 0.3：只改变显示的小数位数。
print(combined == 0.3)  # False：显示相同不代表存储值相同。
print(math.isclose(combined, 0.3))  # True：误差在默认容差内。

0.30000000000000004
0.3
False
True


### 1.2 与零比较时指定绝对容差

abs_tol 是允许的绝对差，默认是 0.0，必须非负。与零比较时，仅靠小于 1 的相对容差，任何非零差值都无法通过；需要按实际单位设置正的 abs_tol。

若只关心绝对差，可以设置 rel_tol=0.0。下面把长度误差的允许范围设为 1e-9 米，这只是本例的要求。

In [2]:
length_error_m = 1e-12
print(math.isclose(length_error_m, 0.0))  # False：默认绝对容差为零。
print(math.isclose(length_error_m, 0.0, rel_tol=0.0, abs_tol=1e-9))
# True：误差不超过本例允许的 1e-9 米。
print(math.isclose(2e-9, 0.0, rel_tol=0.0, abs_tol=1e-9))  # False

False
True
False


## 2 math 的平方根与取整

math 提供实数数学函数。取整时要区分向下、向上和直接舍去小数部分，尤其注意负数。

| API | 中文名称／含义 |
| --- | --- |
| math.sqrt | 平方根；本例使用非负实数输入 |
| math.floor | 向下取整，返回不大于输入的最大整数 |
| math.ceil | 向上取整，返回不小于输入的最小整数 |

这里的向下指朝负无穷方向，不是朝零方向。CPython 对 math.sqrt 的负数输入抛出 ValueError，不会自动返回复数。

In [3]:
print(math.sqrt(81))  # 9.0
print(math.floor(-2.3), math.ceil(-2.3))  # -3 -2

9.0
-3 -2


In [4]:
# 预期 ValueError：直接观察原始异常，之后继续运行下一单元。
# ValueError：超出实数平方根的定义域。
math.sqrt(-1.0)

ValueError: math domain error

## 3 decimal 的十进制精度

### 3.1 从原始十进制文本构造

decimal.Decimal 可以精确表示有限十进制小数。输入本来是十进制文本时，直接交给 Decimal，避免先转为 float。

Decimal(float) 会精确保留该 float 已存储的值，不会恢复输入原本想表达的十进制小数。构造值和后续运算精度是两个问题。

In [5]:
import decimal

text_value = decimal.Decimal("0.1")
float_value = decimal.Decimal(0.1)
print(text_value)  # 0.1：从原始十进制文本构造。
print(float_value)  # 较长小数：精确保留 float 0.1 的二进制近似值。
print(text_value == float_value)  # False
print(text_value + decimal.Decimal("0.2"))  # 0.3

0.1
0.1000000000000000055511151231257827021181583404541015625
False
0.3


### 3.2 用 localcontext 限定运算精度

上下文（context）保存 Decimal 运算的精度和舍入方式。prec 是运算结果的有效数字位数，不是小数点后的位数；默认精度是 28 位。

decimal.localcontext 在 with 内使用当前上下文的副本，退出后恢复原上下文。构造 Decimal 时仍保留输入的全部数字；运算时才按上下文舍入。1/3 的十进制展开无限循环，提高精度也只能得到更长的近似值。

In [6]:
original_precision = decimal.getcontext().prec
with decimal.localcontext() as context:
    context.prec = 4
    measurement = decimal.Decimal("12.3456")
    print(measurement)  # 12.3456：构造时没有按 prec 截断。
    print(measurement + decimal.Decimal("0"))  # 12.35：运算保留 4 位有效数字。
    print(decimal.Decimal("1") / decimal.Decimal("3"))  # 0.3333

print(decimal.getcontext().prec == original_precision)  # True：设置已恢复。

12.3456
12.35
0.3333
True


### 3.3 用 quantize 明确小数位和舍入规则

Decimal.quantize 按另一个 Decimal 的指数确定结果的小数位数。例如 Decimal("0.01") 表示结果保留两位小数。上下文精度仍需足够容纳结果，否则可能触发 InvalidOperation。

| 常量 | 中文名称／含义 |
| --- | --- |
| decimal.ROUND_HALF_EVEN | 舍入到最近值；恰好居中时保留末位为偶数的结果 |
| decimal.ROUND_HALF_UP | 舍入到最近值；恰好居中时向远离零的方向舍入 |

下面输入恰好位于两个百分位数值之间。对于负数，ROUND_HALF_UP 的居中规则同样是远离零。

In [7]:
with decimal.localcontext() as context:
    context.prec = 6
    unit = decimal.Decimal("0.01")
    for text in ("1.245", "-1.245"):
        reading = decimal.Decimal(text)
        even = reading.quantize(unit, rounding=decimal.ROUND_HALF_EVEN)
        away = reading.quantize(unit, rounding=decimal.ROUND_HALF_UP)
        print(even, away)
# 1.24 1.25
# -1.24 -1.25：居中时远离零，不是统一朝正方向舍入。

1.24 1.25
-1.24 -1.25


## 4 fractions 的精确分数

fractions.Fraction 用整数分子和非零整数分母表示有理数，并约分。它可以精确表示 1/3；Decimal 表示这个值时则需要有限精度的近似。

Fraction("0.1") 按十进制文本得到 1/10，Fraction(0.1) 则精确保留 float 的二进制值。limit_denominator 可以限制分母并寻找最近的分数，但这是一种近似选择，不保证恢复原始输入。

In [8]:
import fractions

portion = fractions.Fraction(1, 3)
print(portion + fractions.Fraction(1, 6))  # 1/2：分数运算后约分。
print(portion * 3)  # 1：没有先把 1/3 转成 float。

from_text = fractions.Fraction("0.1")
from_float = fractions.Fraction(0.1)
print(from_text)  # 1/10
print(from_float)  # 3602879701896397/36028797018963968：来自 float 的精确值。
print(from_text == from_float)  # False
print(from_float.limit_denominator(10))  # 1/10：限制分母不超过 10 的近似。

1/2
1
1/10
3602879701896397/36028797018963968
False
1/10


## 5 statistics 的统计口径

### 5.1 平均数、总体方差与样本方差

statistics 可处理小型数值数据集。本例统一使用整数，不混合数值类型。

| API | 中文名称／含义 |
| --- | --- |
| statistics.mean | 算术平均数，即总和除以数据个数 |
| statistics.median | 中位数；排序后居中的值，偶数个数据取中间两个值的平均数 |
| statistics.pvariance | 总体方差，用于描述完整总体的离散程度 |
| statistics.variance | 样本方差，用样本估计总体方差时采用的口径 |

方差衡量数值偏离平均数的程度。设 n 为数据个数；默认由函数计算平均数时，pvariance 把偏差平方和除以 n，variance 则除以 n−1。选择取决于数据代表完整总体还是抽取的样本，不由列表大小自动决定。

In [9]:
import statistics

readings = [2, 4, 6]
print(statistics.mean(readings))  # 4
print(statistics.median(readings))  # 4
print(statistics.median([2, 4, 6, 8]))  # 5.0：中间两个数的平均数。
print(statistics.pvariance(readings))  # 8/3 的浮点近似：偏差平方和为 8。
print(statistics.variance(readings))  # 4：同样的平方和除以 n−1，即 2。

4
4
5.0
2.6666666666666665
4


### 5.2 数据个数不足时不要补造结果

mean、median 和 pvariance 至少需要一个值；variance 至少需要两个值。数据不足时抛出 statistics.StatisticsError。

一个值构成的完整总体方差是零，但一个样本值不能据此得到样本方差；空数据也不能自动当作平均数为零。

In [10]:
print(statistics.pvariance([5]))  # 0：完整总体只有一个值。

0


In [11]:
# 预期 statistics.StatisticsError：直接观察原始异常，之后继续运行下一单元。
# StatisticsError：样本数量不足。
statistics.variance([5])

StatisticsError: variance requires at least two data points

In [12]:
# 预期 statistics.StatisticsError：直接观察原始异常，之后继续运行下一单元。
# StatisticsError：空数据没有算术平均数。
statistics.mean([])

StatisticsError: mean requires at least one data point

## 6 按用途选择随机数

### 6.1 Random 用于可复现的模拟

random 的默认伪随机生成器适合模拟，不适合生成安全凭证。模块级函数共享一个内部生成器；创建 random.Random 实例可以隔离状态。

在相同 Python 版本、相同种子和相同调用顺序下，本例可重放。不要承诺所有采样 API 的结果跨版本不变。randrange(1, 7) 从整数 1 到 6 中选值，不包含右端点 7。

In [13]:
import random

shared_state = random.getstate()
simulation = random.Random(2026)
replay = random.Random(2026)
rolls = [simulation.randrange(1, 7) for _ in range(6)]
replayed_rolls = [replay.randrange(1, 7) for _ in range(6)]

print(rolls)  # 含 6 个 1～6 的整数；同种子、同调用顺序可重放，不承诺跨版本序列。
print(rolls == replayed_rolls)  # True：同种子、同调用顺序重放。
print(random.getstate() == shared_state)  # True：未改变模块级生成器状态。
assert all(1 <= roll <= 6 for roll in rolls)

[1, 3, 5, 5, 6, 1]
True
True


### 6.2 secrets 用于安全随机数

secrets 使用操作系统提供的安全随机源，适合安全令牌等用途。这里刻意不要求可复现，也不设置种子；其系统随机源不能通过 seed 重放序列。

secrets.randbelow 的参数是正整数上界，返回不包含该上界的非负整数。secrets.token_hex 的参数是随机字节数，每个字节编码为两个十六进制字符。

检查长度、字符范围和数值范围即可，不检查某个固定随机值，也不把“两次一定不同”作为通过条件。

In [14]:
import secrets

random_index = secrets.randbelow(10)
demo_token = secrets.token_hex(16)

assert 0 <= random_index < 10
assert len(demo_token) == 32
assert all(character in "0123456789abcdef" for character in demo_token)
print(0 <= random_index < 10, len(demo_token))  # True 32
# 检查性质即可；本例不输出随机令牌，也不要求下一次生成相同值。

True 32


## 7 日期与时间间隔

### 7.1 用 date 计算日历日期

datetime 是模块名，datetime.date 表示年月日；datetime.datetime 则同时包含日期和时刻。仅处理按天安排的事项时，可以先使用 date。

date.fromisoformat 可解析本例的 ISO 8601 日期文本。date 加上 timedelta 得到新日期，两个 date 相减得到 timedelta。本例只使用整数天；date 运算会忽略 timedelta 中的秒和微秒部分。

In [15]:
import datetime

start_date = datetime.date.fromisoformat("2024-02-28")
end_date = start_date + datetime.timedelta(days=2)
print(end_date.isoformat())  # 2024-03-01：跨过闰年的 2 月 29 日。
print((end_date - start_date).days)  # 2
print(start_date.isoformat())  # 2024-02-28：原日期未被修改。

2024-03-01
2
2024-02-28


### 7.2 total_seconds 才是整个间隔的秒数

timedelta 表示时间间隔，可由天、小时、分钟等构造，内部归一化为天、秒和微秒。seconds 属性只是去除整天后的秒部分，不是总秒数。

total_seconds 返回整个间隔的秒数，适合本章这些短间隔。对很大的间隔，它可能丢失微秒精度。

In [16]:
work_interval = datetime.timedelta(days=1, hours=2, minutes=30)
print(work_interval)  # 1 day, 2:30:00
print(work_interval.seconds)  # 9000：只包含整天以外的部分。
print(work_interval.total_seconds())  # 95400.0：包括前面的整天。
assert work_interval.total_seconds() == 26.5 * 3600

1 day, 2:30:00
9000
95400.0


## 8 datetime 的解析与时区信息

### 8.1 区分无时区与带时区对象

无时区对象（naive）缺少足够的时区信息，其时间属于 UTC 还是某地时间，需要由程序约定。带时区对象（aware）的 tzinfo 不为 None，且 utcoffset() 不返回 None，才能把时刻放到共同的时间线上。

datetime.fromisoformat 可解析本例的 ISO 8601 日期时间；带 +00:00 偏移的文本得到带时区对象。isoformat 可输出相应文本。datetime.UTC 是标准库提供的 UTC 时区对象。

固定的其他文本格式可用 strptime 解析。下面格式中的 %Y、%m、%d、%H、%M 分别表示四位年份、月份、日、24 小时制小时和分钟；格式没有时区字段，解析结果仍是 naive。

In [17]:
plain_time = datetime.datetime.strptime("2024/03/01 09:30", "%Y/%m/%d %H:%M")
utc_time = datetime.datetime.fromisoformat("2024-03-01T09:30:00+00:00")

print(plain_time.isoformat())  # 2024-03-01T09:30:00：没有偏移信息。
print(plain_time.tzinfo is None)  # True
print(utc_time.isoformat())  # 2024-03-01T09:30:00+00:00
print(utc_time.tzinfo is datetime.UTC)  # True

2024-03-01T09:30:00
True
2024-03-01T09:30:00+00:00
True


### 8.2 转换时区和附加时区是不同操作

datetime.timezone 表示固定 UTC 偏移；下面的偏移量是正 8 小时。aware 对象调用 astimezone 会调整显示的年月日时分，保持同一个时刻。

replace(tzinfo=...) 只替换时区标签，不转换时间字段，只有已知原始时间属于哪个时区时才能这样附加。不要用它把未知本地时间直接“改成 UTC”。

沿用上一单元的 utc_time 和 plain_time。一个 aware 对象和一个 naive 对象不能直接相减，否则抛出 TypeError。

In [18]:
fixed_east = datetime.timezone(datetime.timedelta(hours=8))
converted_time = utc_time.astimezone(fixed_east)
print(converted_time.isoformat())  # 2024-03-01T17:30:00+08:00
print(converted_time == utc_time)  # True：表示同一时刻。

# 这里只观察重新贴标签的区别；原值实际已知属于 UTC。
retagged_time = utc_time.replace(tzinfo=fixed_east)
print(retagged_time.isoformat())  # 2024-03-01T09:30:00+08:00
print(retagged_time == utc_time)  # False：时间字段没转换，所指时刻改变。

2024-03-01T17:30:00+08:00
True
2024-03-01T09:30:00+08:00
False


In [19]:
# 预期 TypeError：直接观察原始异常，之后继续运行下一单元。
# TypeError：时区信息不足，不能直接相减。
utc_time - plain_time

TypeError: can't subtract offset-naive and offset-aware datetimes

## 9 zoneinfo 与夏令时边界

### 9.1 地区时区不等于固定偏移

zoneinfo.ZoneInfo 用 America/New_York 这样的 IANA 时区标识加载地区规则，其中包含历史 UTC 偏移和夏令时变化。固定偏移 timezone 则不随日期调整。

zoneinfo 优先读取系统时区数据库，找不到时使用第一方 tzdata 包；Windows 通常需要该包。两种数据来源都不可用时，构造 ZoneInfo 会抛出 ZoneInfoNotFoundError。本例需要 America/New_York 数据，不能用固定偏移替代后继续宣称验证了地区规则。

下面使用两个固定 UTC 输入，比较纽约地区规则与固定负 5 小时的显示结果。

In [20]:
import zoneinfo

new_york = zoneinfo.ZoneInfo("America/New_York")
fixed_west = datetime.timezone(datetime.timedelta(hours=-5))
for text in ("2016-01-15T12:00:00+00:00", "2016-07-15T12:00:00+00:00"):
    instant = datetime.datetime.fromisoformat(text)
    print(instant.astimezone(new_york).isoformat())
    print(instant.astimezone(fixed_west).isoformat())
# 一月：两者都显示 2016-01-15T07:00:00-05:00。
# 七月：纽约显示 2016-07-15T08:00:00-04:00，固定偏移仍显示 07:00:00-05:00。

2016-01-15T07:00:00-05:00
2016-01-15T07:00:00-05:00
2016-07-15T08:00:00-04:00
2016-07-15T07:00:00-05:00


### 9.2 fold 区分回拨后的重复时刻

夏令时结束等偏移回拨会让一段当地钟表时间出现两次。fold=0 表示较早的那次，fold=1 表示较晚的那次；只写“当地一点半”不足以区分它们。

沿用 new_york。下面从 2016 年 11 月 6 日两个相差一小时的固定 UTC 时刻转换，观察美国东部回拨时的重复一点半。astimezone 会根据时区规则设置 fold，无需猜测或手工指定当前偏移。

In [21]:
earlier_utc = datetime.datetime(2016, 11, 6, 5, 30, tzinfo=datetime.UTC)
later_utc = earlier_utc + datetime.timedelta(hours=1)
earlier_local = earlier_utc.astimezone(new_york)
later_local = later_utc.astimezone(new_york)

print(earlier_local.isoformat(), earlier_local.fold)
print(later_local.isoformat(), later_local.fold)
# 2016-11-06T01:30:00-04:00 0
# 2016-11-06T01:30:00-05:00 1
assert earlier_local.utcoffset() == datetime.timedelta(hours=-4)
assert later_local.utcoffset() == datetime.timedelta(hours=-5)

2016-11-06T01:30:00-04:00 0
2016-11-06T01:30:00-05:00 1


### 9.3 实际耗时先统一到 UTC

两个 aware datetime 若引用同一个 tzinfo 对象，直接相减会忽略时区偏移；相等比较也忽略这时的 tzinfo 和 fold。因此，当地钟表读数之差不一定是实际经过的时间。

需要实际耗时时，先把两端转换到 UTC 再相减。沿用上一单元的 earlier_local 和 later_local，观察同一个重复小时中的两次一点半。

In [22]:
print(earlier_local.tzinfo is later_local.tzinfo)  # True：同一个时区对象。
print(earlier_local == later_local)  # True：同 tzinfo 时忽略 fold。
print((later_local - earlier_local).total_seconds())  # 0.0：当地读数之差。

elapsed = (
    later_local.astimezone(datetime.UTC)
    - earlier_local.astimezone(datetime.UTC)
)
print(elapsed.total_seconds())  # 3600.0：两个时刻实际相隔一小时。
assert elapsed == datetime.timedelta(hours=1)

True
True
0.0
3600.0


## 本章小结

（1）float 适合近似计算；比较时按任务设置容差，与零比较尤其注意 abs_tol。显示位数不是存储精度。

（2）Decimal 从文本保留十进制输入，用 localcontext 控制运算精度，用 quantize 指定小数位与舍入。Fraction 保留有理数；从 float 构造两者都不会恢复原始十进制意图。

（3）统计函数需要正确的数据口径和最低数量。模拟用独立 Random 和种子；安全用途用 secrets，只检查结果性质。

（4）date 表示日期，timedelta 表示间隔。datetime 的 naive/aware 区别决定能否定位时刻；timezone 是固定偏移，ZoneInfo 是地区规则。

（5）遇到重复的当地时间，应确认 fold 和偏移。实际耗时先统一到 UTC，整个间隔的秒数用 total_seconds。

自查：能否说明“显示一样的数”“相同种子的序列”和“相同的当地时间”分别还需要核对什么条件？

## 练习

（1）先预测下面三个布尔结果，再运行核对，并分别用“构造输入”“相对与绝对容差”“有理数精确运算”说明原因。核对标准是预测与实际输出一致，且能说明把第一行的字符串参数换成相同 float 后，比较关系如何变化。

In [23]:
print(decimal.Decimal("0.2") == decimal.Decimal(0.2))
print(math.isclose(1e-10, 0.0, rel_tol=0.0, abs_tol=1e-9))
print(fractions.Fraction(1, 7) * 7 == 1)
# 先在运行前写下预测；运行后逐项核对输入类型和判定规则。

False
True
True


（2）用种子 73 创建独立 Random，模拟 8 次掷骰子；再创建同种子的独立实例重放。输出平均数、中位数和样本方差，并说明为何这个模拟不适合生成安全凭证。

检查两个列表相同、长度为 8、每项在 1 到 6 之间，模块级随机状态未改变，样本方差非负。若把这 8 个结果视为完整总体，改用 pvariance；两种方差应满足“样本方差乘 7 近似等于总体方差乘 8”。不要用 secrets 来满足重放要求。

In [24]:
exercise_seed = 73
# 在这里创建独立生成器、采样和重放，并用 statistics 计算所需指标。
# 用 random.getstate() 比较调用前后的模块级状态。
# 用 math.isclose 核对两个统计口径之间的关系。

（3）纽约当地时间 2016 年 11 月 6 日 00:30 开始的任务，在当天 02:30 结束。用 ZoneInfo 构造两个 aware datetime，分别计算当地读数差和实际耗时。

检查开始与结束的 UTC 偏移分别为负 4 小时和负 5 小时，当地读数差为 7200 秒，转换到 UTC 后的耗时为 10800 秒。再加上 1 天的间隔，检查 seconds 与 total_seconds 的区别。说明为何不能给两端直接贴同一个固定偏移。

In [25]:
exercise_zone = zoneinfo.ZoneInfo("America/New_York")
# 在这里构造当天的 00:30 和 02:30，并比较直接相减与 UTC 相减。
# 对实际耗时加 datetime.timedelta(days=1)，检查总秒数包含这一天。

### 提示

第一题关注 Decimal 接收的是文本还是已有浮点值。第二题用两个独立实例分别采样，不在每次掷骰子时重新创建生成器。第三题先分别转换两端，再相减；seconds 不包含整天。

### 参考解析

第一题为 False、True、True。Decimal 文本构造值与 float 的精确展开不同；绝对差 1e-10 在 1e-9 容差内；Fraction 的有理数乘法得到整数 1。只把第一行字符串换为 float 0.2 后，两侧保留同一个浮点值，比较变为 True。

第二题在本章 Python 版本下，两次得到 [3, 1, 5, 4, 5, 2, 4, 3]，均值和中位数分别为 3.375 与 3.5，样本方差约 1.982142857，完整总体方差为 1.734375。两种口径的偏差平方和相同，分别乘 7 与 8 即可核对；模拟生成器可按种子重放，因此不用于安全凭证。

第三题开始偏移为 -04:00，结束为 -05:00；当地读数相差两小时，但经过回拨，UTC 时间线相差三小时。实际耗时加一天后，seconds 为 10800，total_seconds() 为 97200.0。固定偏移无法表达当天发生的偏移变化。

## 参考与引用来源

| 网站 | 本章参考内容与定位 |
| --- | --- |
| Python 官方文档（3.12） | [浮点表示与显示误差](https://docs.python.org/3.12/tutorial/floatingpoint.html#representation-error)；[近似比较与零附近容差](https://docs.python.org/3.12/library/math.html#math.isclose)、[平方根](https://docs.python.org/3.12/library/math.html#math.sqrt)、[向下取整](https://docs.python.org/3.12/library/math.html#math.floor)、[向上取整](https://docs.python.org/3.12/library/math.html#math.ceil)、[math 页末的异常行为说明](https://docs.python.org/3.12/library/math.html#constants)；[Decimal 构造与 float 转换](https://docs.python.org/3.12/library/decimal.html#decimal.Decimal)、[上下文精度](https://docs.python.org/3.12/library/decimal.html#decimal.Context)、[默认上下文](https://docs.python.org/3.12/library/decimal.html#decimal.DefaultContext)、[localcontext 的进入和恢复](https://docs.python.org/3.12/library/decimal.html#decimal.localcontext)、[getcontext](https://docs.python.org/3.12/library/decimal.html#decimal.getcontext)、[quantize 与精度限制](https://docs.python.org/3.12/library/decimal.html#decimal.Decimal.quantize)、[居中取偶](https://docs.python.org/3.12/library/decimal.html#decimal.ROUND_HALF_EVEN)、[居中远离零](https://docs.python.org/3.12/library/decimal.html#decimal.ROUND_HALF_UP)；[Fraction 构造与精确值](https://docs.python.org/3.12/library/fractions.html#fractions.Fraction)、[限制分母的近似](https://docs.python.org/3.12/library/fractions.html#fractions.Fraction.limit_denominator)；[统计输入类型](https://docs.python.org/3.12/library/statistics.html)、[平均数与最低数量](https://docs.python.org/3.12/library/statistics.html#statistics.mean)、[中位数](https://docs.python.org/3.12/library/statistics.html#statistics.median)、[总体方差](https://docs.python.org/3.12/library/statistics.html#statistics.pvariance)、[样本方差](https://docs.python.org/3.12/library/statistics.html#statistics.variance)；[random 模块的共享生成器与用途](https://docs.python.org/3.12/library/random.html)、[独立 Random](https://docs.python.org/3.12/library/random.html#random.Random)、[randrange 的右端点](https://docs.python.org/3.12/library/random.html#random.randrange)、[随机状态](https://docs.python.org/3.12/library/random.html#random.getstate)、[重放与版本边界](https://docs.python.org/3.12/library/random.html#notes-on-reproducibility)、[系统随机源不按种子重放](https://docs.python.org/3.12/library/random.html#random.SystemRandom)、[secrets 的安全随机用途](https://docs.python.org/3.12/library/secrets.html#random-numbers)、[randbelow](https://docs.python.org/3.12/library/secrets.html#secrets.randbelow)、[token_hex 编码长度](https://docs.python.org/3.12/library/secrets.html#secrets.token_hex)；[date 运算与不变性](https://docs.python.org/3.12/library/datetime.html#date-objects)、[日期解析](https://docs.python.org/3.12/library/datetime.html#datetime.date.fromisoformat)、[timedelta 的归一化](https://docs.python.org/3.12/library/datetime.html#datetime.timedelta)、[总秒数](https://docs.python.org/3.12/library/datetime.html#datetime.timedelta.total_seconds)、[naive 与 aware](https://docs.python.org/3.12/library/datetime.html#aware-and-naive-objects)、[aware 判定条件](https://docs.python.org/3.12/library/datetime.html#determining-if-an-object-is-aware-or-naive)、[UTC](https://docs.python.org/3.12/library/datetime.html#datetime.UTC)、[ISO 日期时间解析](https://docs.python.org/3.12/library/datetime.html#datetime.datetime.fromisoformat)、[ISO 日期时间输出](https://docs.python.org/3.12/library/datetime.html#datetime.datetime.isoformat)、[strptime](https://docs.python.org/3.12/library/datetime.html#datetime.datetime.strptime)、[格式代码](https://docs.python.org/3.12/library/datetime.html#strftime-and-strptime-format-codes)、[固定偏移时区](https://docs.python.org/3.12/library/datetime.html#timezone-objects)、[时区转换和附加标签](https://docs.python.org/3.12/library/datetime.html#datetime.datetime.astimezone)、[replace](https://docs.python.org/3.12/library/datetime.html#datetime.datetime.replace)、[fold](https://docs.python.org/3.12/library/datetime.html#datetime.datetime.fold)、[datetime 相减与相等比较规则](https://docs.python.org/3.12/library/datetime.html#datetime-objects)、[tzinfo 示例：2016 年美国东部夏令时回拨](https://docs.python.org/3.12/library/datetime.html#tzinfo-objects)；[ZoneInfo 的地区规则与 fold 转换](https://docs.python.org/3.12/library/zoneinfo.html#using-zoneinfo)、[系统数据库与 tzdata 来源](https://docs.python.org/3.12/library/zoneinfo.html#data-sources)。 |